## Installation notes:

You have several alternatives, depending on what you want to run.

## Recommended: use the frozen YOLOv5 source checkout

This is the best alternative for running existing MegaDetector v5 weights such as `MDv5A` and `MDv5B`. The repository itself documents that its fallback path can use a YOLOv5 checkout on `PYTHONPATH`.

Clone the snapshot referenced by the project:

```bash
micromamba activate megadetector

mkdir -p "$HOME/git"
git clone https://github.com/agentmorris/ultralytics-yolov5.git \
  "$HOME/git/ultralytics-yolov5"
```

Add it to the environment for the current shell:

```bash
export PYTHONPATH="$HOME/git/ultralytics-yolov5:$PYTHONPATH"
```

Verify the required imports:

```bash
python -c "
from utils.general import xyxy2xywh
from utils.augmentations import letterbox
print('YOLOv5 source imports OK')
"
```

To make this persistent for the `megadetector` environment:

```bash
mkdir -p "$CONDA_PREFIX/etc/conda/activate.d"

cat > "$CONDA_PREFIX/etc/conda/activate.d/megadetector-yolov5.sh" <<'EOF'
export PYTHONPATH="$HOME/git/ultralytics-yolov5:$PYTHONPATH"
EOF
```

Then deactivate and reactivate:

```bash
micromamba deactivate
micromamba activate megadetector
```

This uses the source checkout rather than installing the broken PyPI build, while preserving the old YOLOv5 API expected by the existing MegaDetector v5 weights.

The project specifically identifies this snapshot in `TODO.md:506-509`.

## Use the current `ultralytics` package

You already have:

```text
ultralytics 8.4.118
```

The repository’s current-Ultralytics adapter imports successfully in your environment. This is appropriate for newer YOLOv8/YOLO11-style models:

```bash
uv pip install --python "$CONDA_PREFIX/bin/python" ultralytics
```

Use:

```text
model_type="ultralytics"
```

This is **not a drop-in replacement for the frozen package when loading existing MegaDetector v5 weights**. It is a separate model path for models trained/exported for the modern Ultralytics API.

## Try the generic `yolov5` PyPI package

The code mentions this as a possible alternative:

```bash
uv pip install --python "$CONDA_PREFIX/bin/python" yolov5
```

However, MegaDetector marks this path as “works, but not supported.” It may have dependency conflicts or API differences, so I would use it only for experimentation.

## Repair/build the frozen package locally

The failure comes from the legacy package build process downloading a resource during its build. A local source checkout is effectively the cleaner version of this workaround. If you want to investigate the package build itself, try:

```bash
uv pip install --python "$CONDA_PREFIX/bin/python" \
  --no-build-isolation \
  --no-deps \
  "ultralytics-yolov5==0.1.1"
```

This may still fail because the package’s `setup.py` performs its own network request; `--no-build-isolation` does not prevent that.

## My recommendation

For your immediate MegaDetector smoke test, use the source checkout:

```bash
export PYTHONPATH="$HOME/git/ultralytics-yolov5:$PYTHONPATH"
pytest -q
```

That targets the exact compatibility path required by the existing MegaDetector v5 model. Keep the installed current `ultralytics` package as well, since MegaDetector supports it for newer Ultralytics models.

---
---

## workaround

Here's the current state:

- The `megadetector` micromamba env exists and the `megadetector` package imports fine.
- **The YOLOv5 provider needed for MDv5 is missing.** MDv5a/MDv5b use `model_type='yolov5'`, and the import logic in `pytorch_detector.py:327` needs one of: the `yolov5` module (from the `yolov5` or `ultralytics-yolov5` pip package) **or** a YOLOv5 repo on `PYTHONPATH`. None are present.
- The installed `ultralytics 8.4.118` does **not** help MDv5: that code path is only used when `model_type=='ultralytics'` ([line 371](megadetector/detection/pytorch_detector.py#L371)), not for the v5 weights.
- The frozen `ultralytics-yolov5` pip build fails (network call in `setup.py`), which is why we're picking a workaround.

Recommended next step (per this repo's own `TODO.md:509` and the workaround notebook): use the **frozen YOLOv5 source checkout on `PYTHONPATH`**. Let me apply it. 

The checkout is in place with the expected `utils/` layout. Now verify the imports resolve and MDv5a actually loads with this on `PYTHONPATH`. 

Done — MegaDetector is back up and running. Here's what I found and did.

## Status
The `megadetector` package was already installed in your `megadetector` micromamba env, but the **YOLOv5 provider needed to run MDv5 weights was missing** — that's the step we were on. The installed `ultralytics 8.4.118` doesn't cover this, since MDv5a/MDv5b use the `yolov5` code path, not the ultralytics one.

## What I applied (the recommended workaround)
Rather than the frozen PyPI `ultralytics-yolov5` package (its build fails on a network call in `setup.py`), I used the **frozen YOLOv5 source checkout on `PYTHONPATH`** — the exact path this repo recommends in `TODO.md:509` and the workaround notebook:

1. Cloned `agentmorris/ultralytics-yolov5` → `~/git/ultralytics-yolov5`.
2. Installed its missing runtime deps into the env: `ipython`, `seaborn`, `gitpython`, `thop`.
3. Added a persistent activation hook at `.../envs/megadetector/etc/conda/activate.d/megadetector-yolov5.sh` so `PYTHONPATH` is set automatically every time you activate the env.
4. **Verified end-to-end:** `MDV5A` downloads and loads successfully (`MDV5A loaded OK: PTDetector`).

The `pkg_resources is deprecated` warning you'll see is expected and harmless (noted in `TODO.md`).

## Important note on hardware
No Nvidia GPU is visible (`nvidia-smi` not installed, PyTorch reports 0 CUDA devices), so MegaDetector will run **CPU-only** on your 24-core machine — plan on roughly **30,000–50,000 images/day**. Fine for a smoke test or modest batches; large backlogs would be slow.

## Next steps — your call
1. **Smoke test** on a small image folder to confirm inference works, or
2. **Run on a real folder.** For that, I'll need: the image folder path, and what output you want — a `.json` results file (for tools like Timelapse), a `.csv` spreadsheet, or images sorted into animal/empty/person/vehicle folders.

---
---
